# `mpl_figure_saver` — save & reopen matplotlib figures (MATLAB-`.fig`-style)

This notebook demonstrates the essential capabilities and the core algorithm.

**The problem.** A matplotlib `savefig` gives you a *picture*. When you later need
the numbers, a different y-scale, or one more annotation, you re-run the whole
analysis. `mpl_figure_saver` makes every save also drop two sidecar files:

| file | what it is | reopen with | survives matplotlib upgrades? |
|------|------------|-------------|-------------------------------|
| `.png`/`.pdf`/… | the image you asked for | — | — |
| `.mplpkl` | a pickle of the **exact** figure | `loadfig()` | no (pickle) |
| `_data.npz` | the **arrays** behind every artist | `replot_from_data()` / `load_arrays()` | **yes** |

So you get a fast, pixel-perfect reopen (pickle) *and* a version-proof fallback
(arrays). Both from one `savefig` or one click of the toolbar **Save** button.


In [ ]:
import matplotlib
# For a real session use:  %matplotlib qt   or   %matplotlib widget
# Here we stay headless so the notebook runs anywhere (CI / SSH).
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

import mpl_figure_saver as mfs

OUT = Path("nb_out"); OUT.mkdir(exist_ok=True)

## 1. Install

One call patches `fig.savefig()` and the Qt / ipympl toolbar **Save** buttons,
and sets configuration. It's safe to re-run (e.g. under `%autoreload`).

- `save_dir` — only required for the widget **`timestamp`** mode (headless/SSH).
- `widget_mode='timestamp'` writes `plot_<timestamp>/` folders; `'dialog'` opens
  a local save dialog (use on a laptop / remote-desktop, not over bare SSH).
- In **VS Code**, call `mfs.fix_vscode_widget_style()` once so `%matplotlib widget`
  figures follow your theme instead of showing a white box.


In [ ]:
mfs.install(save_dir=OUT / "widget_saves", widget_mode="timestamp")
# mfs.fix_vscode_widget_style()   # uncomment inside VS Code

## 2. Save once — get image + arrays + pickle

Nothing special in your plotting code; just call `savefig` as usual.
Note the `cos` line has **no** `label=` and the `axvline` is a reference marker —
both are captured correctly (a bug in naive extractors).

In [ ]:
x = np.linspace(0, 2*np.pi, 200)
fig, ax = plt.subplots()
ax.plot(x, np.sin(x), label="sin")
ax.plot(x, np.cos(x))                 # unlabeled — still captured
ax.axvline(np.pi, color="k", ls="--") # reference line — captured separately
ax.set(xlabel="phase", ylabel="amplitude", title="Trig")
ax.legend()
fig.savefig(OUT / "trig.png")

sorted(p.name for p in OUT.glob("trig.*"))

## 3. Reopen the **exact** figure (pickle)

`loadfig()` returns a live, editable figure with the original layout intact.
It also returns `extra_data` — a free-form dict you can attach at save time
(see §7); empty here because the automatic save path doesn't set it.

In [ ]:
fig2, extra = mfs.loadfig(OUT / "trig.mplpkl")
fig2.axes[0].set_title("Trig (reopened & edited)")
print("extra_data:", extra)
fig2.axes[0].get_title()

## 4. The algorithm — version-proof replot from arrays

`replot_from_data()` ignores the pickle and rebuilds the figure from the `.npz`.
Why this matters: a pickle made with matplotlib 3.8 may not unpickle on 3.12, but
plain NumPy arrays always load. The rebuild covers lines, scatter, errorbars,
bars, images, `axvline`/`axhline`, text, scales, limits, legend, label/tick
colors, figure size, **`fill_between`/`fill_betweenx`**, and **`twinx`/`twiny`**.

### How the two hard cases are reconstructed

**`fill_between`** keeps no source arrays on the artist, so we read them back
from the polygon path. For a band over *n* points the path has a predictable
shape — lower edge left→right, then upper edge right→left — so with the path's
`t_direction` flag (vertical vs horizontal) the original `x, y1, y2` come back,
including `where=` gaps (stored as `NaN`-separated segments).

**Twin axes** are detected via matplotlib's `_twinned_axes` sibling group; the
earliest axes is the *primary* and the other is tagged `twin_of=<primary>` with
`twin_kind='x'` (twinx) or `'y'` (twiny). On replot the primary is drawn, then
the twin is re-attached with `.twinx()`/`.twiny()` — so you get overlaid axes,
not two stacked panels.


In [ ]:
figt, axL = plt.subplots()
axL.plot(x, np.sin(x), "b", label="sin")
axL.fill_between(x, np.sin(x)-0.15, np.sin(x)+0.15, color="b", alpha=0.2)
axL.set_ylabel("sin", color="b"); axL.tick_params(axis="y", labelcolor="b")
axR = axL.twinx()
axR.plot(x, np.exp(-x/3), "r")
axR.set_ylabel("decay", color="r"); axR.tick_params(axis="y", labelcolor="r")
figt.savefig(OUT / "twin_fill.png")

data = mfs.load_plot_data(OUT / "twin_fill.npz")
print("axes:", len(data["axes"]),
      "| twin_of:", [a["twin_of"] for a in data["axes"]],
      "| fills on axis 0:", len(data["axes"][0]["fills"]))

fig_rebuilt = mfs.replot_from_data(data)   # twin + shaded band, from arrays
[a.yaxis.label.get_color() for a in fig_rebuilt.axes]

## 5. Reuse the arrays yourself — `get_series`

When you want to replot the data *your own way* (not via `replot_from_data`),
`get_series` flattens each axes into a uniform list where **every** item — line,
scatter, errorbar, or fill band — has the same `x` and `y` keys. So one loop
plots everything; a fill just additionally carries `y_upper`.

In [ ]:
series = mfs.get_series(OUT / "twin_fill.npz")   # one list per axes

fig5, ax5 = plt.subplots()
for s in series[0]:                              # first axes
    if s["kind"] == "fill":
        ax5.fill_between(s["x"], s["y"], s["y_upper"], alpha=0.2, label=s["label"])
    else:
        ax5.plot(s["x"], s["y"], label=s["label"] or s["kind"])
ax5.legend()

# Grab a single curve's raw x/y directly, no digging through nested dicts:
first = series[0][0]
print(f"{first['kind']} '{first['label']}': "
      f"x[:3]={np.round(first['x'][:3], 3)}, y[:3]={np.round(first['y'][:3], 3)}")

## 6. Regular grids keep their shape

A `2x2` figure stores each panel's grid geometry, so `replot_from_data` rebuilds
a `2x2` — not a `4x1` column. (An irregular `gridspec` falls back to a column;
use `loadfig` on the `.mplpkl` when you need an irregular layout pixel-perfect.)

In [ ]:
figg, axs = plt.subplots(2, 2, figsize=(7, 5))
for k, a in enumerate(axs.ravel()):
    a.plot(x, np.sin(x + k)); a.set_title(f"panel {k}")
figg.savefig(OUT / "grid.png")

figg2 = mfs.replot_from_data(OUT / "grid.npz")
figg2.axes[0].get_subplotspec().get_geometry()[:2]   # -> (2, 2)

## 7. `ExperimentResult` — data + analysis + figure in one file

Bundle raw data, parameters and figures into a single `.expr` for archiving.
`extra_data` on a plain figure save works the same way via
`savefig_bundle(fig, path, extra_data={...})`, recovered through `loadfig`.

> **Security:** `.mplpkl` and `.expr` use `pickle` — only load files you trust.
> The `.npz` is plain arrays and always safe.

In [ ]:
r = mfs.ExperimentResult("ESR #42", params={"B_mT": 12.5, "power_dBm": -10})
r.store("freq_GHz", np.linspace(2.80, 2.95, 200))
r.store("signal", np.random.default_rng(0).normal(size=200))
r.attach_figure(figt)
r.save(OUT / "esr_042.expr")

r2 = mfs.ExperimentResult.load(OUT / "esr_042.expr")
r2.summary()

## What needs a real GUI (not shown headless)

- **Qt toolbar Save** (`%matplotlib qt`): dialog gains two checkboxes —
  *Also save plot data (.npz)* and *Also save editable figure (.mplpkl)* — and
  writes a `<name>/` folder. Unticking both = plain image, as before.
- **ipympl Save** (`%matplotlib widget`): `dialog` mode opens a local Python
  dialog (nothing written if you cancel); `timestamp` mode writes
  `plot_<timestamp>[_suptitle]/` into `save_dir` and lets the browser download
  proceed — the right choice over SSH.
